# **Celebal Excellence Internship Program 2026**

### **Week 7: Delta Lake Assignment**
### **Submitted By: Manjit Bajaj**


## **Delta Lake MERGE Implementation (Incremental Data Processing)**

### **Objective**

The objective of this assignment is to implement incremental data processing using Delta Lake with Apache Spark.

The workflow includes:

- Loading customer data into a Delta table
- Performing data cleaning
- Creating an incremental dataset
- Applying Delta MERGE (UPSERT)
- Validating results
- Displaying the final dataset

### **Technologies Used**

- Python
- Apache Spark (PySpark)
- Delta Lake
- Google Colab

Dataset:
Superstore Dataset (Customer Data)

In [2]:
import pyspark
print(pyspark.__version__)

3.5.1


In [3]:
!pip uninstall -y dataproc-spark-connect
!pip install pyspark==3.5.1

Found existing installation: dataproc-spark-connect 1.1.0
Uninstalling dataproc-spark-connect-1.1.0:
  Successfully uninstalled dataproc-spark-connect-1.1.0


In [4]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

In [5]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

In [6]:
builder = (
    SparkSession.builder
    .appName("DeltaLakeAssignment")
    .master("local[*]")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.5.1


## Step 1: Load Dataset

In [10]:
from google.colab import files

uploaded = files.upload()

Saving Sample - Superstore.csv to Sample - Superstore.csv


In [11]:
import os
print(os.listdir())

['.config', 'Sample - Superstore.csv', 'sample_data']


In [16]:
# read dataset using spark
customer_df = spark.read.csv(
    "Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

In [17]:
# display dataset
customer_df.show(10, truncate=False)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+----------------------------------------------------------------+--------+--------+--------+--------+
|Row ID|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name  |Segment  |Country      |City           |State     |Postal Code|Region|Product ID     |Category       |Sub-Category|Product Name                                                    |Sales   |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+----------------------------------------------------------------+--------+--------+--------+--------+
|1     |CA-2016-152156|11/8/2016 |11/11/2016|Second Class  |CG-12520   |Claire Gute  

In [18]:
# check schema
customer_df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [19]:
# count rows
print("Total Rows:", customer_df.count())

Total Rows: 9994


In [20]:
# display columns
print(customer_df.columns)

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [21]:
# create customer master
customer_master = customer_df.select(
    "Customer ID",
    "Customer Name",
    "Segment",
    "City",
    "State",
    "Country"
)

In [22]:
customer_master.show(10, truncate=False)

+-----------+---------------+---------+---------------+----------+-------------+
|Customer ID|Customer Name  |Segment  |City           |State     |Country      |
+-----------+---------------+---------+---------------+----------+-------------+
|CG-12520   |Claire Gute    |Consumer |Henderson      |Kentucky  |United States|
|CG-12520   |Claire Gute    |Consumer |Henderson      |Kentucky  |United States|
|DV-13045   |Darrin Van Huff|Corporate|Los Angeles    |California|United States|
|SO-20335   |Sean O'Donnell |Consumer |Fort Lauderdale|Florida   |United States|
|SO-20335   |Sean O'Donnell |Consumer |Fort Lauderdale|Florida   |United States|
|BH-11710   |Brosina Hoffman|Consumer |Los Angeles    |California|United States|
|BH-11710   |Brosina Hoffman|Consumer |Los Angeles    |California|United States|
|BH-11710   |Brosina Hoffman|Consumer |Los Angeles    |California|United States|
|BH-11710   |Brosina Hoffman|Consumer |Los Angeles    |California|United States|
|BH-11710   |Brosina Hoffman

In [23]:
customer_master.coalesce(1).write.mode("overwrite").option(
    "header", True
).csv("delta-lake-assignment/data/customer_master")

# Step 2: Data Cleaning

In this step we will:

- Check the dataset
- Handle missing values
- Remove duplicate records
- Validate the cleaned data
- Save the cleaned data as a Delta Table

In [24]:
# read customer master table
customer_master = spark.read.csv(
    "delta-lake-assignment/data/customer_master",
    header=True,
    inferSchema=True
)

customer_master.show(5, truncate=False)

+-----------+---------------+---------+---------------+----------+-------------+
|Customer ID|Customer Name  |Segment  |City           |State     |Country      |
+-----------+---------------+---------+---------------+----------+-------------+
|CG-12520   |Claire Gute    |Consumer |Henderson      |Kentucky  |United States|
|CG-12520   |Claire Gute    |Consumer |Henderson      |Kentucky  |United States|
|DV-13045   |Darrin Van Huff|Corporate|Los Angeles    |California|United States|
|SO-20335   |Sean O'Donnell |Consumer |Fort Lauderdale|Florida   |United States|
|SO-20335   |Sean O'Donnell |Consumer |Fort Lauderdale|Florida   |United States|
+-----------+---------------+---------+---------------+----------+-------------+
only showing top 5 rows



In [25]:
customer_master.printSchema()

root
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)



In [26]:
print("Total Records Before Cleaning:", customer_master.count())

Total Records Before Cleaning: 9994


In [27]:
# check for null values
from pyspark.sql.functions import col, when, count

customer_master.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in customer_master.columns
]).show()

+-----------+-------------+-------+----+-----+-------+
|Customer ID|Customer Name|Segment|City|State|Country|
+-----------+-------------+-------+----+-----+-------+
|          0|            0|      0|   0|    0|      0|
+-----------+-------------+-------+----+-----+-------+



In [28]:
# handle missing values
customer_master = customer_master.fillna({
    "Customer Name": "Unknown",
    "City": "Unknown",
    "State": "Unknown",
    "Country": "Unknown",
    "Segment": "Unknown"
})

In [29]:
customer_master.show(5)

+-----------+---------------+---------+---------------+----------+-------------+
|Customer ID|  Customer Name|  Segment|           City|     State|      Country|
+-----------+---------------+---------+---------------+----------+-------------+
|   CG-12520|    Claire Gute| Consumer|      Henderson|  Kentucky|United States|
|   CG-12520|    Claire Gute| Consumer|      Henderson|  Kentucky|United States|
|   DV-13045|Darrin Van Huff|Corporate|    Los Angeles|California|United States|
|   SO-20335| Sean O'Donnell| Consumer|Fort Lauderdale|   Florida|United States|
|   SO-20335| Sean O'Donnell| Consumer|Fort Lauderdale|   Florida|United States|
+-----------+---------------+---------+---------------+----------+-------------+
only showing top 5 rows



In [30]:
print("Rows Before Removing Duplicates:", customer_master.count())

Rows Before Removing Duplicates: 9994


In [31]:
customer_master_clean = customer_master.dropDuplicates(["Customer ID"])

In [32]:
print("Rows After Removing Duplicates:", customer_master_clean.count())

Rows After Removing Duplicates: 793


In [33]:
# display clean dataset
customer_master_clean.show(10, truncate=False)

+-----------+--------------------+-----------+-------------+--------------+-------------+
|Customer ID|Customer Name       |Segment    |City         |State         |Country      |
+-----------+--------------------+-----------+-------------+--------------+-------------+
|AA-10315   |Alex Avila          |Consumer   |Minneapolis  |Minnesota     |United States|
|AA-10375   |Allen Armold        |Consumer   |Mesa         |Arizona       |United States|
|AA-10480   |Andrew Allen        |Consumer   |Concord      |North Carolina|United States|
|AA-10645   |Anna Andreadi       |Consumer   |Chester      |Pennsylvania  |United States|
|AB-10015   |Aaron Bergman       |Consumer   |Seattle      |Washington    |United States|
|AB-10060   |Adam Bellavance     |Home Office|New York City|New York      |United States|
|AB-10105   |Adrian Barton       |Consumer   |Phoenix      |Arizona       |United States|
|AB-10150   |Aimee Bixby         |Consumer   |Long Beach   |New York      |United States|
|AB-10165 

In [34]:
from pyspark.sql.functions import count

duplicates = customer_master_clean.groupBy("Customer ID") \
    .count() \
    .filter("count > 1")

duplicates.show()

+-----------+-----+
|Customer ID|count|
+-----------+-----+
+-----------+-----+



In [38]:
#rename columns
from pyspark.sql.functions import col

customer_master_clean = customer_master_clean.select(
    col("Customer ID").alias("customer_id"),
    col("Customer Name").alias("customer_name"),
    col("Segment").alias("segment"),
    col("City").alias("city"),
    col("State").alias("state"),
    col("Country").alias("country")
)

In [39]:
customer_master_clean.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- segment: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- country: string (nullable = false)



In [40]:
# create data table folder
delta_path = "delta-lake-assignment/delta_customer"

In [41]:
customer_master_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

In [42]:
# read delta table
delta_customer = spark.read \
    .format("delta") \
    .load(delta_path)

In [43]:
# count records
print("Total Records in Delta Table:", delta_customer.count())

Total Records in Delta Table: 793


In [44]:
# display records
delta_customer.show(10, truncate=False)

+-----------+--------------------+-----------+-------------+--------------+-------------+
|customer_id|customer_name       |segment    |city         |state         |country      |
+-----------+--------------------+-----------+-------------+--------------+-------------+
|AA-10315   |Alex Avila          |Consumer   |Minneapolis  |Minnesota     |United States|
|AA-10375   |Allen Armold        |Consumer   |Mesa         |Arizona       |United States|
|AA-10480   |Andrew Allen        |Consumer   |Concord      |North Carolina|United States|
|AA-10645   |Anna Andreadi       |Consumer   |Chester      |Pennsylvania  |United States|
|AB-10015   |Aaron Bergman       |Consumer   |Seattle      |Washington    |United States|
|AB-10060   |Adam Bellavance     |Home Office|New York City|New York      |United States|
|AB-10105   |Adrian Barton       |Consumer   |Phoenix      |Arizona       |United States|
|AB-10150   |Aimee Bixby         |Consumer   |Long Beach   |New York      |United States|
|AB-10165 

## Part 3: Create Incremental Dataset

Objective

We'll simulate a new day's customer data where:

- Some existing customers have updated information (e.g., city or segment changed).
- Some are completely new customers.

In [45]:
# select existing customer to update
existing_customers = delta_customer.limit(5)

existing_customers.show(truncate=False)

+-----------+-------------+--------+-----------+--------------+-------------+
|customer_id|customer_name|segment |city       |state         |country      |
+-----------+-------------+--------+-----------+--------------+-------------+
|AA-10315   |Alex Avila   |Consumer|Minneapolis|Minnesota     |United States|
|AA-10375   |Allen Armold |Consumer|Mesa       |Arizona       |United States|
|AA-10480   |Andrew Allen |Consumer|Concord    |North Carolina|United States|
|AA-10645   |Anna Andreadi|Consumer|Chester    |Pennsylvania  |United States|
|AB-10015   |Aaron Bergman|Consumer|Seattle    |Washington    |United States|
+-----------+-------------+--------+-----------+--------------+-------------+



In [46]:
# create updates records
from pyspark.sql.functions import lit

updated_customers = existing_customers \
    .withColumn("city", lit("Pune")) \
    .withColumn("segment", lit("Corporate"))

updated_customers.show(truncate=False)

+-----------+-------------+---------+----+--------------+-------------+
|customer_id|customer_name|segment  |city|state         |country      |
+-----------+-------------+---------+----+--------------+-------------+
|AA-10315   |Alex Avila   |Corporate|Pune|Minnesota     |United States|
|AA-10375   |Allen Armold |Corporate|Pune|Arizona       |United States|
|AA-10480   |Andrew Allen |Corporate|Pune|North Carolina|United States|
|AA-10645   |Anna Andreadi|Corporate|Pune|Pennsylvania  |United States|
|AB-10015   |Aaron Bergman|Corporate|Pune|Washington    |United States|
+-----------+-------------+---------+----+--------------+-------------+



In [47]:
# create new customers
new_customer_data = [
    ("CUST9001", "Manjit Bajaj", "Consumer", "Ahmednagar", "Maharashtra", "India"),
    ("CUST9002", "Rahul Sharma", "Corporate", "Pune", "Maharashtra", "India"),
    ("CUST9003", "Priya Patel", "Home Office", "Mumbai", "Maharashtra", "India"),
    ("CUST9004", "Amit Kumar", "Consumer", "Nagpur", "Maharashtra", "India"),
    ("CUST9005", "Sneha Joshi", "Corporate", "Nashik", "Maharashtra", "India")
]

In [48]:
new_customers = spark.createDataFrame(
    new_customer_data,
    delta_customer.columns
)

new_customers.show(truncate=False)

+-----------+-------------+-----------+----------+-----------+-------+
|customer_id|customer_name|segment    |city      |state      |country|
+-----------+-------------+-----------+----------+-----------+-------+
|CUST9001   |Manjit Bajaj |Consumer   |Ahmednagar|Maharashtra|India  |
|CUST9002   |Rahul Sharma |Corporate  |Pune      |Maharashtra|India  |
|CUST9003   |Priya Patel  |Home Office|Mumbai    |Maharashtra|India  |
|CUST9004   |Amit Kumar   |Consumer   |Nagpur    |Maharashtra|India  |
|CUST9005   |Sneha Joshi  |Corporate  |Nashik    |Maharashtra|India  |
+-----------+-------------+-----------+----------+-----------+-------+



In [49]:
# combine updates and new records
incremental_df = updated_customers.union(new_customers)
incremental_df.show(truncate=False)

+-----------+-------------+-----------+----------+--------------+-------------+
|customer_id|customer_name|segment    |city      |state         |country      |
+-----------+-------------+-----------+----------+--------------+-------------+
|AA-10315   |Alex Avila   |Corporate  |Pune      |Minnesota     |United States|
|AA-10375   |Allen Armold |Corporate  |Pune      |Arizona       |United States|
|AA-10480   |Andrew Allen |Corporate  |Pune      |North Carolina|United States|
|AA-10645   |Anna Andreadi|Corporate  |Pune      |Pennsylvania  |United States|
|AB-10015   |Aaron Bergman|Corporate  |Pune      |Washington    |United States|
|CUST9001   |Manjit Bajaj |Consumer   |Ahmednagar|Maharashtra   |India        |
|CUST9002   |Rahul Sharma |Corporate  |Pune      |Maharashtra   |India        |
|CUST9003   |Priya Patel  |Home Office|Mumbai    |Maharashtra   |India        |
|CUST9004   |Amit Kumar   |Consumer   |Nagpur    |Maharashtra   |India        |
|CUST9005   |Sneha Joshi  |Corporate  |N

In [50]:
# count records
print("Incremental Records:", incremental_df.count())

Incremental Records: 10


In [51]:
# save incremental dataset
incremental_path = "delta-lake-assignment/data/customer_incremental"

incremental_df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(incremental_path)

In [52]:
# read incremental data
incremental_df = spark.read.csv(
    incremental_path,
    header=True,
    inferSchema=True
)

In [53]:
incremental_df.show(truncate=False)

+-----------+-------------+-----------+----------+--------------+-------------+
|customer_id|customer_name|segment    |city      |state         |country      |
+-----------+-------------+-----------+----------+--------------+-------------+
|AA-10315   |Alex Avila   |Corporate  |Pune      |Minnesota     |United States|
|AA-10375   |Allen Armold |Corporate  |Pune      |Arizona       |United States|
|AA-10480   |Andrew Allen |Corporate  |Pune      |North Carolina|United States|
|AA-10645   |Anna Andreadi|Corporate  |Pune      |Pennsylvania  |United States|
|AB-10015   |Aaron Bergman|Corporate  |Pune      |Washington    |United States|
|CUST9001   |Manjit Bajaj |Consumer   |Ahmednagar|Maharashtra   |India        |
|CUST9002   |Rahul Sharma |Corporate  |Pune      |Maharashtra   |India        |
|CUST9003   |Priya Patel  |Home Office|Mumbai    |Maharashtra   |India        |
|CUST9004   |Amit Kumar   |Consumer   |Nagpur    |Maharashtra   |India        |
|CUST9005   |Sneha Joshi  |Corporate  |N

In [55]:
# compare data
print("Delta table")
delta_customer.show(5)
print("Incremental table")
incremental_df.show(10)

Delta table
+-----------+-------------+--------+-----------+--------------+-------------+
|customer_id|customer_name| segment|       city|         state|      country|
+-----------+-------------+--------+-----------+--------------+-------------+
|   AA-10315|   Alex Avila|Consumer|Minneapolis|     Minnesota|United States|
|   AA-10375| Allen Armold|Consumer|       Mesa|       Arizona|United States|
|   AA-10480| Andrew Allen|Consumer|    Concord|North Carolina|United States|
|   AA-10645|Anna Andreadi|Consumer|    Chester|  Pennsylvania|United States|
|   AB-10015|Aaron Bergman|Consumer|    Seattle|    Washington|United States|
+-----------+-------------+--------+-----------+--------------+-------------+
only showing top 5 rows

Incremental table
+-----------+-------------+-----------+----------+--------------+-------------+
|customer_id|customer_name|    segment|      city|         state|      country|
+-----------+-------------+-----------+----------+--------------+-------------+
|  

In [56]:
# temporary views
delta_customer.createOrReplaceTempView("customer_master")
incremental_df.createOrReplaceTempView("customer_incremental")

In [57]:
spark.sql("""
SELECT *
FROM customer_incremental
""").show()

+-----------+-------------+-----------+----------+--------------+-------------+
|customer_id|customer_name|    segment|      city|         state|      country|
+-----------+-------------+-----------+----------+--------------+-------------+
|   AA-10315|   Alex Avila|  Corporate|      Pune|     Minnesota|United States|
|   AA-10375| Allen Armold|  Corporate|      Pune|       Arizona|United States|
|   AA-10480| Andrew Allen|  Corporate|      Pune|North Carolina|United States|
|   AA-10645|Anna Andreadi|  Corporate|      Pune|  Pennsylvania|United States|
|   AB-10015|Aaron Bergman|  Corporate|      Pune|    Washington|United States|
|   CUST9001| Manjit Bajaj|   Consumer|Ahmednagar|   Maharashtra|        India|
|   CUST9002| Rahul Sharma|  Corporate|      Pune|   Maharashtra|        India|
|   CUST9003|  Priya Patel|Home Office|    Mumbai|   Maharashtra|        India|
|   CUST9004|   Amit Kumar|   Consumer|    Nagpur|   Maharashtra|        India|
|   CUST9005|  Sneha Joshi|  Corporate| 

## Part 4 – Delta Lake MERGE

In [58]:
# import delta table
from delta.tables import DeltaTable

In [59]:
# load delta table
delta_path = "delta-lake-assignment/delta_customer"

delta_table = DeltaTable.forPath(spark, delta_path)

In [60]:
# check master table
print("Before MERGE")
delta_table.toDF().show(10, truncate=False)

Before MERGE
+-----------+--------------------+-----------+-------------+--------------+-------------+
|customer_id|customer_name       |segment    |city         |state         |country      |
+-----------+--------------------+-----------+-------------+--------------+-------------+
|AA-10315   |Alex Avila          |Consumer   |Minneapolis  |Minnesota     |United States|
|AA-10375   |Allen Armold        |Consumer   |Mesa         |Arizona       |United States|
|AA-10480   |Andrew Allen        |Consumer   |Concord      |North Carolina|United States|
|AA-10645   |Anna Andreadi       |Consumer   |Chester      |Pennsylvania  |United States|
|AB-10015   |Aaron Bergman       |Consumer   |Seattle      |Washington    |United States|
|AB-10060   |Adam Bellavance     |Home Office|New York City|New York      |United States|
|AB-10105   |Adrian Barton       |Consumer   |Phoenix      |Arizona       |United States|
|AB-10150   |Aimee Bixby         |Consumer   |Long Beach   |New York      |United State

In [61]:
# check row count befor merge
before_count = delta_table.toDF().count()
print("Rows Before Merge:", before_count)

Rows Before Merge: 793


In [62]:
# perfroming merge
(
    delta_table.alias("target")
    .merge(
        incremental_df.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdate(
        set={
            "customer_name": "source.customer_name",
            "segment": "source.segment",
            "city": "source.city",
            "state": "source.state",
            "country": "source.country"
        }
    )
    .whenNotMatchedInsert(
        values={
            "customer_id": "source.customer_id",
            "customer_name": "source.customer_name",
            "segment": "source.segment",
            "city": "source.city",
            "state": "source.state",
            "country": "source.country"
        }
    )
    .execute()
)

In [63]:
merged_df = spark.read.format("delta").load(delta_path)

In [64]:
merged_df.show(20, truncate=False)

+-----------+--------------------+-----------+---------------+--------------+-------------+
|customer_id|customer_name       |segment    |city           |state         |country      |
+-----------+--------------------+-----------+---------------+--------------+-------------+
|AA-10315   |Alex Avila          |Corporate  |Pune           |Minnesota     |United States|
|AA-10375   |Allen Armold        |Corporate  |Pune           |Arizona       |United States|
|AA-10480   |Andrew Allen        |Corporate  |Pune           |North Carolina|United States|
|AA-10645   |Anna Andreadi       |Corporate  |Pune           |Pennsylvania  |United States|
|AB-10015   |Aaron Bergman       |Corporate  |Pune           |Washington    |United States|
|AB-10060   |Adam Bellavance     |Home Office|New York City  |New York      |United States|
|AB-10105   |Adrian Barton       |Consumer   |Phoenix        |Arizona       |United States|
|AB-10150   |Aimee Bixby         |Consumer   |Long Beach     |New York      |Uni

In [65]:
after_count = merged_df.count()
print("Rows After Merge:", after_count)

Rows After Merge: 798


In [66]:
# check update customers
merged_df.filter(
    merged_df.customer_id.isin(
        [
            "CG-12520",
            "DV-13045"
        ]
    )
).show(truncate=False)

+-----------+---------------+---------+-----------+----------+-------------+
|customer_id|customer_name  |segment  |city       |state     |country      |
+-----------+---------------+---------+-----------+----------+-------------+
|CG-12520   |Claire Gute    |Consumer |Henderson  |Kentucky  |United States|
|DV-13045   |Darrin Van Huff|Corporate|Los Angeles|California|United States|
+-----------+---------------+---------+-----------+----------+-------------+



In [67]:
# check new customers
merged_df.filter(
    merged_df.customer_id.startswith("CUST")
).show(truncate=False)

+-----------+-------------+-----------+----------+-----------+-------+
|customer_id|customer_name|segment    |city      |state      |country|
+-----------+-------------+-----------+----------+-----------+-------+
|CUST9001   |Manjit Bajaj |Consumer   |Ahmednagar|Maharashtra|India  |
|CUST9002   |Rahul Sharma |Corporate  |Pune      |Maharashtra|India  |
|CUST9003   |Priya Patel  |Home Office|Mumbai    |Maharashtra|India  |
|CUST9004   |Amit Kumar   |Consumer   |Nagpur    |Maharashtra|India  |
|CUST9005   |Sneha Joshi  |Corporate  |Nashik    |Maharashtra|India  |
+-----------+-------------+-----------+----------+-----------+-------+



In [68]:
# save final output
merged_df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("delta-lake-assignment/output/final_customer_data")